In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

X_train_sm = pd.read_csv("../data/processed/X_train_smote.csv")
y_train_sm = pd.read_csv("../data/processed/y_train_smote.csv").squeeze()
X_test = pd.read_csv("../data/processed/X_test_scaled.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("Train:", X_train_sm.shape, "| Test:", X_test.shape)
print("Train classes:", y_train_sm.value_counts().to_dict())
print("Test classes:", y_test.value_counts().to_dict())

Train: (1972, 53) | Test: (294, 53)
Train classes: {0: 986, 1: 986}
Test classes: {0: 247, 1: 47}


Training first model (Logistic Regression)

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)

y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

results = {}
results["Logistic Regression"] = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
}

print(pd.DataFrame(results).T.round(3))

                     Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression     0.861        0.6   0.383  0.468    0.813


In [3]:
cols = ["MonthlyIncome", "Age", "IncomeToExperience"]
print("TRAIN")
print(X_train_sm[cols].describe().round(2))
print()
print("TEST")
print(X_test[cols].describe().round(2))
print()
print("Same columns in train and test:", list(X_train_sm.columns) == list(X_test.columns))

TRAIN
       MonthlyIncome      Age  IncomeToExperience
count        1972.00  1972.00             1972.00
mean           -0.18    -0.20                0.14
std             0.91     0.99                1.11
min            -1.19    -2.07               -1.75
25%            -0.83    -0.87               -0.64
50%            -0.48    -0.33               -0.04
75%             0.09     0.44                0.62
max             2.89     2.51                4.66

TEST
       MonthlyIncome     Age  IncomeToExperience
count         294.00  294.00              294.00
mean           -0.04   -0.04               -0.01
std             1.06    0.98                1.04
min            -1.18   -2.07               -1.67
25%            -0.79   -0.76               -0.86
50%            -0.43   -0.22               -0.17
75%             0.30    0.55                0.62
max             2.89    2.40                3.28

Same columns in train and test: True


In [5]:
import os, time

p = "../data/processed/X_train_smote.csv"
print("File last saved:", time.ctime(os.path.getmtime(p)))

chk = pd.read_csv(p)
print("Shape:", chk.shape)
print("IncomeToExperience mean:", round(chk["IncomeToExperience"].mean(), 2))
print("IncomeToExperience std:", round(chk["IncomeToExperience"].std(), 2))

File last saved: Mon Oct 19 12:20:13 2026
Shape: (1972, 53)
IncomeToExperience mean: 0.14
IncomeToExperience std: 1.11


Training the second model (Random Forest)

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

results["Random Forest"] = {
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf),
    "Recall": recall_score(y_test, y_pred_rf),
    "F1": f1_score(y_test, y_pred_rf),
    "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
}

print(pd.DataFrame(results).T.round(3))

                     Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression     0.861      0.600   0.383  0.468    0.813
Random Forest           0.827      0.409   0.191  0.261    0.739


Training the third model (XGBoost)

In [8]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=3,
    random_state=42, eval_metric="logloss"
)
xgb.fit(X_train_sm, y_train_sm)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

results["XGBoost"] = {
    "Accuracy": accuracy_score(y_test, y_pred_xgb),
    "Precision": precision_score(y_test, y_pred_xgb),
    "Recall": recall_score(y_test, y_pred_xgb),
    "F1": f1_score(y_test, y_pred_xgb),
    "ROC-AUC": roc_auc_score(y_test, y_prob_xgb),
}

print(pd.DataFrame(results).T.round(3))

                     Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression     0.861      0.600   0.383  0.468    0.813
Random Forest           0.827      0.409   0.191  0.261    0.739
XGBoost                 0.854      0.591   0.277  0.377    0.750


Comparing the 3 models fairly (cross-validation)

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

X_tr = pd.read_csv("../data/processed/X_train_scaled.csv")
y_tr = pd.read_csv("../data/processed/y_train.csv").squeeze()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                             random_state=42, eval_metric="logloss"),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = {}

for name, model in models.items():
    pipe = ImbPipeline([("smote", SMOTE(random_state=42)), ("model", model)])
    s = cross_validate(pipe, X_tr, y_tr, cv=cv,
                       scoring=["roc_auc", "recall", "precision", "f1"])
    cv_rows[name] = {
        "ROC-AUC": s["test_roc_auc"].mean(),
        "Recall": s["test_recall"].mean(),
        "Precision": s["test_precision"].mean(),
        "F1": s["test_f1"].mean(),
    }

print(pd.DataFrame(cv_rows).T.round(3))

c:\Users\ranou\OneDrive\Desktop\attrition-project\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\ranou\OneDrive\Desktop\attrition-project\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\ranou\OneDrive\Desktop\attrition-project\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\r

                     ROC-AUC  Recall  Precision     F1
Logistic Regression    0.833   0.495      0.758  0.596
Random Forest          0.769   0.332      0.632  0.430
XGBoost                0.794   0.395      0.599  0.470


In [10]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import cross_val_predict

lr_pipe = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

oof_prob = cross_val_predict(lr_pipe, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]

rows = []
for t in np.arange(0.20, 0.80, 0.05):
    pred = (oof_prob >= t).astype(int)
    rows.append({
        "Threshold": round(t, 2),
        "Precision": precision_score(y_tr, pred, zero_division=0),
        "Recall": recall_score(y_tr, pred),
        "F1": f1_score(y_tr, pred),
    })

print(pd.DataFrame(rows).round(3).to_string(index=False))

 Threshold  Precision  Recall    F1
      0.20      0.364   0.737 0.487
      0.25      0.442   0.705 0.544
      0.30      0.496   0.663 0.568
      0.35      0.563   0.637 0.598
      0.40      0.617   0.584 0.600
      0.45      0.689   0.547 0.610
      0.50      0.758   0.495 0.599
      0.55      0.804   0.411 0.544
      0.60      0.819   0.358 0.498
      0.65      0.855   0.311 0.456
      0.70      0.909   0.263 0.408
      0.75      0.902   0.195 0.320
      0.80      0.933   0.147 0.255


In [12]:
from sklearn.metrics import confusion_matrix

threshold = 0.35
y_pred_t = (y_prob >= threshold).astype(int)

print("Threshold:", threshold)
print("Precision:", round(precision_score(y_test, y_pred_t), 3))
print("Recall:   ", round(recall_score(y_test, y_pred_t), 3))
print("F1:       ", round(f1_score(y_test, y_pred_t), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, y_prob), 3))
print()
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred_t),
    index=["Actually stayed", "Actually left"],
    columns=["Predicted stay", "Predicted leave"],
)
print(cm)

Threshold: 0.35
Precision: 0.511
Recall:    0.511
F1:        0.511
ROC-AUC:   0.813

                 Predicted stay  Predicted leave
Actually stayed             224               23
Actually left                23               24


In [13]:
import joblib

joblib.dump(lr, "../models/logistic_regression.pkl")
joblib.dump(0.35, "../models/threshold.pkl")

pd.DataFrame(results).T.round(3).to_csv("../models/model_comparison.csv")

print("Model and threshold saved")

Model and threshold saved
